# Tutorial 2 — Package a parallel LC resonator as a reusable Composite

## Prerequisites

Complete [the primitive
resonator](../simple_resonator/01_model_author.qmd). You should know
primitive parts, complete `net()` calls, the canonical ground, logical
Ports, and authoring diagrams.

## Objects introduced here

An immutable custom `Library`, `CompositePlan`, child components, public
`ParameterSpec` and `ParameterRef` values, `expose_pin()`, `build()`,
and the resulting `ComponentInstance`.

## What the reader will build

A reusable one-pin parallel-LC component and an outer one-port circuit
that uses it through a coupling capacitor.

## What inspection or Result is produced

The composite’s public parameter handles and the outer authoring
diagram. The current V1 scaffold deliberately stops at construction,
before numerical simulation evidence exists.

## Notice the repetition

Tutorial 1 had to repeat a capacitor, inductor, shared top node, and
grounded returns everywhere that it needed this resonator. That topology
is one stable thing, so repeating it in every outer circuit invites a
wiring mismatch and leaks an implementation detail to consumers.

## Give the package one immutable Library

A custom `Library` is an ordinary immutable Python object, not a mutable
Notebook registry. Its factory starts a `CompositePlan`; that private
plan owns the child topology before it seals one `ComponentInstance`.

## Declare public parameters with baselines

The composite exposes capacitance and inductance deliberately. Each
`ParameterSpec` records the preferred unit and each `baseline` is the
sealed physical value. The returned `ParameterRef` is then what the
child primitive receives, and later what a model author may deliberately
select for an optimization; unexposed child details remain private.

## Build the child topology and expose just one terminal

The factory adds primitive children, nets their upper terminals, grounds
their returns to the eventual outer Plan ground, exposes the shared
upper node as one external pin, then calls `build()`. There is no
composite-local ground.

``` python
"""Reusable parallel-LC composite used by the simple-resonator tutorials."""

from __future__ import annotations

from scnsim import (
    ComponentInstance,
    CompositePlan,
    Library,
    ParameterSpec,
    library as sc,
    units as u,
)


class ResonatorLibrary(Library):
    """Immutable catalog containing the tutorial's reusable LC package."""

    def parallel_linear_lc_resonator(
        self,
        *,
        id: str,
        capacitance: object,
        inductance: object,
    ) -> ComponentInstance:
        """Package a grounded parallel capacitor and inductor behind one pin."""

        component = CompositePlan(id=id, library=self)
        resonator_capacitance = component.parameter(
            id="capacitance",
            baseline=capacitance,
            spec=ParameterSpec(unit=u.fF),
        )
        resonator_inductance = component.parameter(
            id="inductance",
            baseline=inductance,
            spec=ParameterSpec(unit=u.nH),
        )
        capacitor = component.add(
            sc.capacitor(
                id="capacitor",
                capacitance=resonator_capacitance,
            )
        )
        inductor = component.add(
            sc.inductor(
                id="inductor",
                inductance=resonator_inductance,
            )
        )
        terminal = component.net(
            capacitor.pin("terminal_1"),
            inductor.pin("terminal_1"),
        )
        component.ground(
            capacitor.pin("terminal_2"),
            inductor.pin("terminal_2"),
        )
        component.expose_pin(id="terminal", at=terminal)
        return component.build()


library = object.__new__(ResonatorLibrary)
"""Immutable custom Library object exported by this module."""
```

## Import the same package for use

The source above is the only definition. An executable cell imports that
exact adjacent module instead of copying its factory into the notebook.

In [ ]:
from resonator_library import library as resonators
from scnsim import CircuitPlan, library as sc, units as u

## Use the package in an outer circuit

The outer Plan only knows the composite’s exposed `terminal` pin. It
does not wire the composite children or repeat their ground attachments.
Its public parameter handles remain inspectable by deliberate name.

In [ ]:
plan = CircuitPlan(id="reusable_simple_resonator")
coupling_cap = plan.add(
    sc.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
resonator = plan.add(
    resonators.parallel_linear_lc_resonator(
        id="resonator",
        capacitance=110.0 * u.fF,
        inductance=5.8 * u.nH,
    )
)
signal_boundary = plan.net(coupling_cap.pin("terminal_1"))
resonator_node = plan.net(
    coupling_cap.pin("terminal_2"),
    resonator.pin("terminal"),
    id="resonator_node",
)
plan.add_port(
    id="signal_in",
    at=signal_boundary,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
resonator.parameter("capacitance").show()

## Inspect the packaged outer circuit

The authoring diagram still shows the physical relationship while the
outer Plan retains a small, reusable wiring surface.

In [ ]:
from scnsim import CircuitDiagramSpec

diagram = plan.render_schematic(
    CircuitDiagramSpec(show_parameter_values=True)
)
diagram.show()

## Recognize the equivalent built-in package

SCNSim’s built-in grounded-LC factory is the same kind of packaging
result: it also exposes one `terminal` while keeping the parallel
primitive branches and their ground group inside the component. Use it
when the built-in semantics are the model you want; author a custom
composite when your reusable topology is your team’s responsibility.

``` python
resonator = plan.add(
    sc.grounded_parallel_linear_lc_resonator(
        id="resonator",
        capacitance=110.0 * u.fF,
        inductance=5.8 * u.nH,
    )
)
```

## Learned objects

You packaged repeated primitive topology in an immutable custom
`Library`, exposed only supported parameters and a terminal, and used
the resulting component in an outer `CircuitPlan`.

Next: [consume the team-owned resonator
model](../simple_resonator/02_model_user.qmd).